In [ ]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
                        text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

# DATA 266 Lab 2 — Part 2: Fine-Tuning Stable Diffusion 1.5 with LoRA

Two controlled runs on the same data, split, base model, steps and evaluation; the **only** difference is LoRA rank/alpha (A: 4/4, B: 64/64).

Notebook order: run information (cell 1, required first) → **configuration (the single source of every parameter)** → data preparation → fine-tuning → generation → evaluation → Hugging Face artifacts → safety → appendix: smoke test (gated, off by default).

Real training and evaluation are **not implemented yet** (`RUN_TRAINING` / `RUN_EVALUATION` stay `False`). Decisions and experiments are logged in `DECISIONS.md`.

## 0. Configuration

In [ ]:
# ===================== CONFIGURATION (single source of truth) =====================
import os
from pathlib import Path

EXPERIMENT_ID = "UNASSIGNED"      # P2-EXP-### only for real training runs; smoke tests never get an ID (D-046)

# --- Execution flags (real work is off by default) ---
RUN_SMOKE_TEST = True            # P2-L* run on CPU; P2-S* run only when CUDA is available
RUN_TRAINING = False              # real training runs (P2-EXP-###) — not implemented yet
RUN_EVALUATION = False            # generation + metrics + uploads — not implemented yet

# --- Environment & paths ---
try:
    import torch as _torch
    CUDA_AVAILABLE = _torch.cuda.is_available()
except ImportError:
    CUDA_AVAILABLE = False
ON_KAGGLE = Path("/kaggle/input").exists()
DEVICE = "cuda:0"                 # one GPU only (team rule 3)
REPO_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd().parent   # notebook runs from notebooks/
P2_WORK_DIR = Path(os.environ.get("P2_WORK_DIR",   # adapters, generated images, caches: OUTSIDE the repo (D-059)
                   "/kaggle/working/p2_work" if ON_KAGGLE else Path.home() / "data266_lab2_work" / "p2"))
EVAL_DIR = REPO_DIR / "eval"      # created later, when evaluation artifacts are written (D-045)
P2_PROMPTS_PATH = EVAL_DIR / "p2_prompts.csv"
P2_HUMAN_RATINGS_PATH = EVAL_DIR / "p2_human_ratings.csv"   # OPEN: needs approval before creation (D-058)
HF_TOKEN_ENV = "HF_TOKEN"         # read from the environment / Kaggle Secrets — never stored in the repo
HF_REPO_RUN_A = HF_REPO_RUN_B = HF_REPO_IMAGES = None   # OPEN: Hugging Face repo names

# --- Dataset (D-047 to D-050, D-064) ---
DATASET_ID = "reach-vb/pokemon-blip-captions"        # DECIDED (D-064): 833 rows, image/text, license stated on the card
DATASET_LICENSE = "cc-by-nc-sa-4.0"                  # as stated on the dataset card; re-verified by P2-L2
DATASET_REJECTED_ID = "lambda/naruto-blip-captions"  # considered; rejected for EXP-001: no license stated on its card
IMAGE_COLUMN, CAPTION_COLUMN = "image", "text"
SPLIT_SEED, TEST_FRACTION = 266, 0.10                # rubric
MIN_IMAGE_SIDE = 256                                 # TUNE: smaller images are removed
NEAR_DUP_HAMMING_MAX = 4                             # TUNE: 64-bit dHash distance for near-duplicates
RESOLUTION = 512                                     # SD 1.5 resolution (rubric)
RESIZE_FILTER = "lanczos"                            # short side -> 512, then center crop
RANDOM_FLIP = False                                  # DECIDED: no augmentation

# --- Base model and method (D-051, D-052) ---
BASE_MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
BASE_MODEL_VARIANT = "fp16"
WEIGHT_DTYPE = "float16"                             # frozen weights on the GPU; LoRA parameters kept in fp32
LORA_TARGET_MODULES = ("to_q", "to_k", "to_v", "to_out.0")   # UNet attention projections only
LORA_DROPOUT, LORA_INIT = 0.0, "gaussian"

# --- Runs: rank/alpha is the ONLY difference between A and B (D-053) ---
RUNS = {"A": {"rank": 4, "alpha": 4}, "B": {"rank": 64, "alpha": 64}}
TRAIN = {                                            # shared by both runs (D-054)
    "learning_rate": 1e-4,                           # TUNE
    "lr_scheduler": "cosine", "warmup_fraction": 0.05,   # TUNE
    "max_train_steps": 1500,                         # TUNE (fixed from the P2-S4 timing before P2-EXP-001)
    "effective_batch": 4, "micro_batch": 4,          # TUNE (micro/accum split from P2-S4 memory)
    "gradient_checkpointing": False,                 # TUNE (from P2-S4)
    "adam_betas": (0.9, 0.999), "adam_weight_decay": 1e-2, "adam_epsilon": 1e-8, "max_grad_norm": 1.0,
    "latent_mode": "online",                         # CANDIDATE: "online" | "cached_moments" | "cached_mean" (D-055)
    "cache_text_embeddings": True,                   # DECIDED: text encoder is frozen
    "seed": 266,
}
LATENT_MODES = ("online", "cached_moments", "cached_mean")

# --- Generation (D-056) ---
GEN = {"scheduler": "DPMSolverMultistep", "steps": 30, "guidance_scale": 7.5, "height": RESOLUTION,
       "width": RESOLUTION, "seeds": (0, 1), "lora_scale": 1.0, "negative_prompt": None, "safety_checker": True}
N_PROMPTS = {"in_domain": 10, "out_of_domain": 5, "compositional": 5}   # rubric
N_GRID_PROMPTS = 8                                                    # rubric

# --- Evaluation (D-057, D-058) ---
CLIP_MODEL_ID = "openai/clip-vit-large-patch14"      # CANDIDATE (CLIP score + memorization embeddings)
LPIPS_NET = "alex"
KID_SUBSETS = 100
MEMORIZATION_N, MEMORIZATION_SHOW = 10, 3            # rubric
HUMAN_EVAL_PAIRS, HUMAN_EVAL_SEED, HUMAN_EVAL_MIN_RATERS = 25, 266, 3   # rubric
HUMAN_EVAL_CRITERIA = ("domain_style", "prompt_match")
VRAM_LIMIT_MB = 16384                                # one 16 GB GPU (team rule 3)

# --- Smoke test (not an experiment) ---
SMOKE_GPU_IMAGES = 8                                 # real images through preprocessing / VAE in P2-S1
SMOKE_TRAIN_STEPS = 3                                # optimizer steps per configuration in P2-S4
SMOKE_MICRO_BATCHES = (4, 2, 1)                      # tried in order until one fits (effective batch unchanged)
SMOKE_RELEASE_TOLERANCE_MB = 64
SMOKE_KID_N = 4                                      # KID plumbing only — never a metric result

print({"experiment_id": EXPERIMENT_ID, "run_smoke_test": RUN_SMOKE_TEST, "run_training": RUN_TRAINING,
       "run_evaluation": RUN_EVALUATION, "cuda": CUDA_AVAILABLE, "dataset": DATASET_ID, "base_model": BASE_MODEL_ID,
       "runs": RUNS, "work_dir": str(P2_WORK_DIR)})

## 1. Data preparation (Task 2.1)

Load → remove corrupted/duplicate images (counts reported) → deterministic 90/10 split with seed 266 → 512×512. Test images are used only for evaluation.

Helpers below are implemented and validated by the smoke test. _Running the full data preparation is not implemented yet._

In [ ]:
# ===== 1. Data preparation helpers (D-048 to D-050) — validated by the smoke test; no experiment runs here =====
import hashlib, io
import numpy as np
import torch
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset

_RESAMPLE = {"lanczos": Image.Resampling.LANCZOS, "bicubic": Image.Resampling.BICUBIC}

def image_bytes(value):
    """Raw encoded bytes from a datasets Image(decode=False) value, raw bytes, or a PIL image."""
    if isinstance(value, dict):
        return value.get("bytes") or (Path(value["path"]).read_bytes() if value.get("path") else None)
    if isinstance(value, (bytes, bytearray)):
        return bytes(value)
    if isinstance(value, Image.Image):
        buf = io.BytesIO(); value.save(buf, format="PNG"); return buf.getvalue()
    return None

def decode_rgb(raw):
    """(RGB image, None) or (None, reason). Transparency is composited on white; EXIF orientation applied."""
    if not raw:
        return None, "missing_image"
    try:
        with Image.open(io.BytesIO(raw)) as probe:
            probe.verify()
        img = Image.open(io.BytesIO(raw)); img.load()
    except Exception:  # noqa: BLE001 — any decoder failure means a corrupted image
        return None, "decode_error"
    img = ImageOps.exif_transpose(img)
    if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
        rgba = img.convert("RGBA")
        img = Image.alpha_composite(Image.new("RGBA", rgba.size, (255, 255, 255, 255)), rgba)
    return img.convert("RGB"), None

def dhash64(img):
    """64-bit difference hash (perceptual) used for near-duplicate detection."""
    g = np.asarray(img.convert("L").resize((9, 8), Image.Resampling.LANCZOS), dtype=np.int16)
    return int("".join("1" if b else "0" for b in (g[:, 1:] > g[:, :-1]).flatten()), 2)

def clean_records(records, min_side=None, hamming_max=None):
    """records: iterable of {"idx", "image", "text"} in dataset order; the first occurrence of a duplicate is kept.
    Returns (kept, removed, report). Kept rows hold raw bytes (decoded lazily) to bound memory."""
    min_side = MIN_IMAGE_SIDE if min_side is None else min_side
    hamming_max = NEAR_DUP_HAMMING_MAX if hamming_max is None else hamming_max
    kept, removed, exact = [], [], {}
    for r in records:
        raw, text = image_bytes(r.get("image")), (r.get("text") or "").strip()
        img, reason = decode_rgb(raw)
        dup_of = None
        if reason is None and min(img.size) < min_side:
            reason = "too_small"
        if reason is None and not text:
            reason = "missing_caption"
        if reason is None:
            px = hashlib.sha256(img.tobytes() + repr(img.size).encode()).hexdigest()
            if px in exact:
                reason, dup_of = "exact_duplicate", exact[px]
        if reason is None:
            h = dhash64(img)
            near = next((k for k in kept if (k["dhash"] ^ h).bit_count() <= hamming_max), None)
            if near is not None:
                reason, dup_of = "near_duplicate", near["idx"]
        if reason is None:
            exact[px] = r["idx"]
            kept.append({"idx": r["idx"], "text": text, "raw": raw, "size": img.size, "dhash": h})
        else:
            removed.append({"idx": r["idx"], "reason": reason, "duplicate_of": dup_of})
    reasons = ["missing_image", "decode_error", "too_small", "missing_caption", "exact_duplicate", "near_duplicate"]
    report = {"total": len(kept) + len(removed), "kept": len(kept), "removed": len(removed),
              **{k: sum(x["reason"] == k for x in removed) for k in reasons}}
    return kept, removed, report

def split_train_test(kept, test_fraction=None, seed=None):
    """Deterministic 90/10 split of the CLEANED rows (dedup happens before the split; D-048, D-049)."""
    test_fraction = TEST_FRACTION if test_fraction is None else test_fraction
    seed = SPLIT_SEED if seed is None else seed
    ids = [k["idx"] for k in kept]
    perm = np.random.default_rng(seed).permutation(len(ids))
    n_test = int(round(test_fraction * len(ids)))
    return sorted(ids[i] for i in perm[n_test:]), sorted(ids[i] for i in perm[:n_test])

def preprocess(img, resolution=None):
    """Short side -> resolution (Lanczos), then center crop; no augmentation (D-050)."""
    resolution = RESOLUTION if resolution is None else resolution
    w, h = img.size
    s = resolution / min(w, h)
    img = img.resize((max(resolution, round(w * s)), max(resolution, round(h * s))), _RESAMPLE[RESIZE_FILTER])
    left, top = (img.width - resolution) // 2, (img.height - resolution) // 2
    return img.crop((left, top, left + resolution, top + resolution))

def to_model_tensor(img):
    """PIL RGB -> float32 CHW tensor in [-1, 1] (VAE input and LPIPS input convention)."""
    return torch.from_numpy(np.asarray(img, dtype=np.float32) / 127.5 - 1.0).permute(2, 0, 1).contiguous()

def tokenize_captions(tokenizer, captions):
    return tokenizer(list(captions), padding="max_length", max_length=tokenizer.model_max_length,
                     truncation=True, return_tensors="pt").input_ids

class CaptionedImages(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return {"pixel_values": to_model_tensor(preprocess(decode_rgb(r["raw"])[0])), "text": r["text"], "idx": r["idx"]}

def collate_batch(batch):
    return {"pixel_values": torch.stack([b["pixel_values"] for b in batch]),
            "text": [b["text"] for b in batch], "idx": [b["idx"] for b in batch]}

def make_loader(rows, batch_size, seed, shuffle=True):
    """Seeded loader; num_workers=0 keeps the sample order reproducible across machines."""
    return DataLoader(CaptionedImages(rows), batch_size=batch_size, shuffle=shuffle, drop_last=True,
                      generator=torch.Generator().manual_seed(seed), collate_fn=collate_batch, num_workers=0)

def load_records(dataset_id=None, limit=None):
    """Full dataset (GPU machine): images left undecoded so corrupted files can be detected and counted."""
    from datasets import Image as HFImage, load_dataset
    ds = load_dataset(dataset_id or DATASET_ID, split="train").cast_column(IMAGE_COLUMN, HFImage(decode=False))
    for i in range(len(ds) if limit is None else min(limit, len(ds))):
        row = ds[i]
        yield {"idx": i, "image": row[IMAGE_COLUMN], "text": row[CAPTION_COLUMN]}

## 2. Fine-tuning (Task 2.2)

Run A (LoRA rank 4) and run B (LoRA rank 64) with identical settings; per-run table of trainable parameters, learning rate, steps, batch size, peak GPU memory and training time.

Helpers below are used by the smoke test. _The training runs themselves (P2-EXP-###) are not implemented yet._

In [ ]:
# ===== 2. Fine-tuning helpers (D-051 to D-055) — used by the smoke test; the training run itself is NOT implemented =====
import contextlib

def run_config(run):
    """Complete configuration of one run: the shared TRAIN settings plus the run's rank/alpha (the only difference)."""
    return {"run": run, **RUNS[run], **TRAIN, "steps": TRAIN["max_train_steps"],
            "grad_accum": TRAIN["effective_batch"] // TRAIN["micro_batch"], "base_model": BASE_MODEL_ID,
            "dataset": DATASET_ID, "target_modules": LORA_TARGET_MODULES, "lora_dropout": LORA_DROPOUT, "lora_init": LORA_INIT}

def torch_dtype(name):
    return getattr(torch, name)

def _variant_kw():
    return {"variant": BASE_MODEL_VARIANT} if BASE_MODEL_VARIANT else {}

def load_sd_components(device=DEVICE, need=("tokenizer", "text_encoder", "vae", "unet", "noise_scheduler")):
    """Load only the SD 1.5 components a stage needs (sequential GPU residency, as in Part 1 D-037)."""
    from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
    from transformers import CLIPTextModel, CLIPTokenizer
    dt, out = torch_dtype(WEIGHT_DTYPE), {}
    if "tokenizer" in need:
        out["tokenizer"] = CLIPTokenizer.from_pretrained(BASE_MODEL_ID, subfolder="tokenizer")
    if "text_encoder" in need:
        out["text_encoder"] = CLIPTextModel.from_pretrained(BASE_MODEL_ID, subfolder="text_encoder", torch_dtype=dt,
                                                            **_variant_kw()).to(device).requires_grad_(False).eval()
    if "vae" in need:
        out["vae"] = AutoencoderKL.from_pretrained(BASE_MODEL_ID, subfolder="vae", torch_dtype=dt,
                                                   **_variant_kw()).to(device).requires_grad_(False).eval()
    if "unet" in need:
        out["unet"] = UNet2DConditionModel.from_pretrained(BASE_MODEL_ID, subfolder="unet", torch_dtype=dt,
                                                           **_variant_kw()).to(device).requires_grad_(False)
    if "noise_scheduler" in need:
        out["noise_scheduler"] = DDPMScheduler.from_pretrained(BASE_MODEL_ID, subfolder="scheduler")
    return out

def add_lora(unet, rank, alpha):
    """Attach a PEFT LoRA adapter to the UNet attention projections; trainable parameters are cast to fp32."""
    from peft import LoraConfig
    unet.requires_grad_(False)
    unet.add_adapter(LoraConfig(r=rank, lora_alpha=alpha, lora_dropout=LORA_DROPOUT, init_lora_weights=LORA_INIT,
                                target_modules=list(LORA_TARGET_MODULES)))
    params = [p for p in unet.parameters() if p.requires_grad]
    for p in params:
        p.data = p.data.float()
    return params

def count_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "trainable_pct": round(100.0 * trainable / total, 4)}

@torch.no_grad()
def encode_text(tokenizer, text_encoder, captions):
    return text_encoder(tokenize_captions(tokenizer, captions).to(text_encoder.device))[0]

@torch.no_grad()
def vae_moments(vae, pixel_values):
    """Posterior mean and std of the VAE latent distribution (cacheable; D-055)."""
    dist = vae.encode(pixel_values.to(vae.device, vae.dtype)).latent_dist
    return dist.mean, dist.std

def latents_from_moments(mean, std, mode, generator, scaling_factor):
    """'online' and 'cached_moments' draw a FRESH posterior sample every time (same distribution as normal VAE
    training); 'cached_mean' is deterministic. No single stochastic sample is ever frozen (D-055)."""
    if mode not in LATENT_MODES:
        raise ValueError(f"unknown latent mode {mode!r}")
    z = mean if mode == "cached_mean" else \
        mean + std * torch.randn(mean.shape, generator=generator, device=mean.device, dtype=mean.dtype)
    return z * scaling_factor

def diffusion_loss(unet, noise_scheduler, latents, text_emb, generator):
    """Standard SD 1.5 epsilon-prediction MSE (as in diffusers train_text_to_image_lora.py)."""
    if noise_scheduler.config.prediction_type != "epsilon":
        raise ValueError(f"unexpected prediction_type {noise_scheduler.config.prediction_type!r}")
    noise = torch.randn(latents.shape, generator=generator, device=latents.device, dtype=latents.dtype)
    t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],),
                      generator=generator, device=latents.device).long()
    pred = unet(noise_scheduler.add_noise(latents, noise, t), t, encoder_hidden_states=text_emb, return_dict=False)[0]
    return torch.nn.functional.mse_loss(pred.float(), noise.float())

def amp_tools(device):
    """fp16 autocast + GradScaler on CUDA; plain fp32 elsewhere (CPU plumbing tests)."""
    on_cuda = str(device).startswith("cuda") and WEIGHT_DTYPE == "float16"
    ctx = (lambda: torch.autocast("cuda", dtype=torch.float16)) if on_cuda else contextlib.nullcontext
    return ctx, torch.amp.GradScaler("cuda", enabled=on_cuda)

def make_optimizer(params, total_steps):
    from diffusers.optimization import get_scheduler
    opt = torch.optim.AdamW(params, lr=TRAIN["learning_rate"], betas=TRAIN["adam_betas"],
                            weight_decay=TRAIN["adam_weight_decay"], eps=TRAIN["adam_epsilon"])
    sched = get_scheduler(TRAIN["lr_scheduler"], optimizer=opt, num_training_steps=total_steps,
                          num_warmup_steps=int(TRAIN["warmup_fraction"] * total_steps))
    return opt, sched

def optimizer_step(unet, params, opt, sched, scaler, autocast, noise_scheduler, micro_batches, generator):
    """One optimizer step over gradient-accumulation micro-batches [(latents, text_emb), ...]."""
    losses = []
    for latents, emb in micro_batches:
        with autocast():
            loss = diffusion_loss(unet, noise_scheduler, latents, emb, generator)
        scaler.scale(loss / len(micro_batches)).backward()
        losses.append(float(loss.detach()))
    scaler.unscale_(opt)
    grad_norm = float(torch.nn.utils.clip_grad_norm_(params, TRAIN["max_grad_norm"]))
    scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
    return float(np.mean(losses)), grad_norm

def save_lora(unet, out_dir):
    """Save the adapter in diffusers format (loadable with pipe.load_lora_weights; Part 3 reuses it)."""
    from diffusers import StableDiffusionPipeline
    from diffusers.utils import convert_state_dict_to_diffusers
    from peft.utils import get_peft_model_state_dict
    StableDiffusionPipeline.save_lora_weights(save_directory=str(out_dir), safe_serialization=True,
                                              unet_lora_layers=convert_state_dict_to_diffusers(get_peft_model_state_dict(unet)))

if RUN_TRAINING:   # real runs (P2-EXP-###) start only after eval/p2_prompts.csv is committed (D-045)
    raise NotImplementedError("Training runs are not implemented yet — keep RUN_TRAINING = False.")

## 3. Image generation (Task 2.3)

20 fixed prompts (10 in-domain from test captions, 5 out-of-domain, 5 compositional) × seeds 0/1 × base, A, B with identical sampler settings; side-by-side grids for 8 prompts.

_Prompt set (`eval/p2_prompts.csv`) and generation runs are not created yet._

In [ ]:
# ===== 3. Prompts and generation helpers (D-056) — the 20-prompt set itself lives in eval/p2_prompts.csv (later) =====
import pandas as pd

PROMPT_COLUMNS = ["prompt_id", "group", "prompt", "source_idx"]

def draw_in_domain_prompts(test_rows, n=None, seed=None):
    """In-domain prompts are TEST-split captions drawn deterministically (rubric 2.3.1)."""
    n = N_PROMPTS["in_domain"] if n is None else n
    pick = np.random.default_rng(SPLIT_SEED if seed is None else seed).choice(len(test_rows), size=n, replace=False)
    return [(test_rows[i]["idx"], test_rows[i]["text"]) for i in sorted(pick)]

def build_prompt_table(in_domain, out_of_domain, compositional):
    rows = [("in_domain", p, src) for src, p in in_domain] + [("out_of_domain", p, None) for p in out_of_domain] \
        + [("compositional", p, None) for p in compositional]
    df = pd.DataFrame([{"prompt_id": f"P{i + 1:02d}", "group": g, "prompt": p, "source_idx": s}
                       for i, (g, p, s) in enumerate(rows)], columns=PROMPT_COLUMNS)
    counts = df["group"].value_counts().to_dict()
    if counts != {k: v for k, v in N_PROMPTS.items() if v}:
        raise ValueError(f"prompt groups {counts} != required {N_PROMPTS}")
    return df

def build_pipeline(lora_dir=None, device=DEVICE):
    """SD 1.5 pipeline with the shared sampler; the built-in safety checker is the NSFW filter (D-056)."""
    from diffusers import DPMSolverMultistepScheduler, StableDiffusionPipeline
    pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL_ID, torch_dtype=torch_dtype(WEIGHT_DTYPE), **_variant_kw())
    if not GEN["safety_checker"]:
        pipe.safety_checker = None
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    if lora_dir is not None:
        pipe.load_lora_weights(str(lora_dir))
    pipe.set_progress_bar_config(disable=True)
    return pipe.to(device)

@torch.no_grad()
def generate(pipe, prompt, seed, lora=False):
    """One image with the shared settings; returns (image, nsfw_flag)."""
    out = pipe(prompt, num_inference_steps=GEN["steps"], guidance_scale=GEN["guidance_scale"],
               height=GEN["height"], width=GEN["width"], negative_prompt=GEN["negative_prompt"],
               generator=torch.Generator(device=pipe.device).manual_seed(seed),
               cross_attention_kwargs={"scale": GEN["lora_scale"]} if lora else None)
    flags = out.nsfw_content_detected
    return out.images[0], (bool(flags[0]) if flags is not None else None)

## 4. Evaluation (Task 2.4)

KID (in-domain generations vs test images; noisy with few images), CLIP score, LPIPS seed diversity, blinded human evaluation (25 pairs, ≥3 raters), GPU-hours and peak memory; baseline (base model) vs runs in one table; two failure modes.

_Evaluation runs are not implemented yet._

In [ ]:
# ===== 4. Evaluation helpers (D-057, D-058) — weight-free parts validated locally; models load only on the GPU machine =====
RESULTS_COLUMNS = ["model", "kid_mean", "kid_std", "kid_subset_size", "clip_score_all", "clip_score_in_domain",
                   "clip_score_out_of_domain", "clip_score_compositional", "lpips_seed_diversity",
                   "human_win_rate_style", "human_win_rate_prompt", "nsfw_flagged", "gpu_hours", "peak_gpu_mb"]
RUN_TABLE_COLUMNS = ["run", "rank", "alpha", "total_params", "trainable_params", "trainable_pct", "learning_rate",
                     "steps", "effective_batch", "micro_batch", "grad_accum", "latent_mode", "peak_gpu_mb",
                     "train_time_s", "gpu_hours"]

def kid_subset_size(n_real, n_fake, requested=None):
    """KID subset size must not exceed the smaller set (20 in-domain generations -> at most 20)."""
    n = min(n_real, n_fake)
    if n < 2:
        raise ValueError("KID needs at least 2 images per set")
    return n if requested is None else min(requested, n)

def to_uint8_batch(images):
    """PIL list -> uint8 tensor [N, 3, H, W] (torchmetrics KID with normalize=False)."""
    return torch.from_numpy(np.stack([np.asarray(im.convert("RGB"), dtype=np.uint8) for im in images])).permute(0, 3, 1, 2)

def kid(real_images, fake_images, subset_size=None, device=DEVICE):
    from torchmetrics.image.kid import KernelInceptionDistance
    m = KernelInceptionDistance(subsets=KID_SUBSETS, normalize=False,
                                subset_size=kid_subset_size(len(real_images), len(fake_images), subset_size)).to(device)
    m.update(to_uint8_batch(real_images).to(device), real=True)
    m.update(to_uint8_batch(fake_images).to(device), real=False)
    mean, std = m.compute()
    return float(mean), float(std)

def clip_score_from_embeddings(img_emb, txt_emb):
    """torchmetrics CLIPScore definition per pair: 100 * max(cosine, 0)."""
    img = torch.nn.functional.normalize(img_emb.float(), dim=-1)
    txt = torch.nn.functional.normalize(txt_emb.float(), dim=-1)
    return (100.0 * (img * txt).sum(-1).clamp(min=0)).tolist()

class ClipModel:
    """One CLIP model for CLIP score and memorization embeddings (loaded once per stage)."""
    def __init__(self, device=DEVICE):
        from transformers import CLIPModel, CLIPProcessor
        self.device = device
        self.model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device).eval()
        self.proc = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
    @torch.no_grad()
    def image_embeddings(self, images):
        x = self.proc(images=images, return_tensors="pt").to(self.device)
        return torch.nn.functional.normalize(self.model.get_image_features(**x).float(), dim=-1)
    @torch.no_grad()
    def text_embeddings(self, texts):
        x = self.proc(text=list(texts), return_tensors="pt", padding=True, truncation=True).to(self.device)
        return torch.nn.functional.normalize(self.model.get_text_features(**x).float(), dim=-1)
    def scores(self, images, prompts):
        return clip_score_from_embeddings(self.image_embeddings(images), self.text_embeddings(prompts))

def lpips_model(device=DEVICE):
    import lpips
    return lpips.LPIPS(net=LPIPS_NET, verbose=False).to(device).eval()

@torch.no_grad()
def lpips_distance(model, img_a, img_b):
    dev = next(model.parameters()).device
    return float(model(to_model_tensor(img_a)[None].to(dev), to_model_tensor(img_b)[None].to(dev)))

def nearest_training(gen_emb, train_emb):
    """Memorization check: for each generated image, the most similar training image (cosine of CLIP embeddings)."""
    sims = gen_emb @ train_emb.T
    best = sims.max(dim=1)
    return best.values.tolist(), best.indices.tolist()

def draw_human_pairs(candidates, n=None, seed=None):
    """Blinded pairs: candidates = [(prompt_id, seed), ...]; final-vs-base side randomised per pair (seeded)."""
    n = HUMAN_EVAL_PAIRS if n is None else n
    rng = np.random.default_rng(HUMAN_EVAL_SEED if seed is None else seed)
    chosen = rng.choice(len(candidates), size=n, replace=False)
    final_left = rng.integers(0, 2, size=n).astype(bool)
    return [{"pair_id": f"H{i + 1:02d}", "prompt_id": candidates[c][0], "seed": int(candidates[c][1]),
             "left": "final" if fl else "base", "right": "base" if fl else "final"}
            for i, (c, fl) in enumerate(zip(chosen, final_left))]

def human_win_rate(ratings, pairs):
    """ratings: rows {rater, pair_id, criterion, choice in {left, right}} -> final-model win rate per criterion."""
    side = {p["pair_id"]: p for p in pairs}
    out = {}
    for crit in HUMAN_EVAL_CRITERIA:
        rows = [r for r in ratings if r["criterion"] == crit]
        wins = sum(side[r["pair_id"]][r["choice"]] == "final" for r in rows)
        out[crit] = {"win_rate": wins / len(rows) if rows else None, "judgments": len(rows),
                     "raters": len({r["rater"] for r in rows})}
    return out

if RUN_EVALUATION:
    raise NotImplementedError("Generation/evaluation runs are not implemented yet — keep RUN_EVALUATION = False.")

## 5. Hugging Face artifacts (Task 2.2.3, 2.3.3)

Both adapters and all generated images (with prompt, seed and model metadata) go to Hugging Face — never to GitHub. Model card for the final run.

_Nothing is uploaded yet; helpers define the required structure._

In [ ]:
# ===== 5. Hugging Face artifacts: model card + image metadata (D-059) — structure only; nothing is uploaded yet =====
MODEL_CARD_FIELDS = ("repo_id", "run", "base_model", "base_model_license", "dataset_id", "dataset_license",
                     "training", "results", "limitations", "intended_use")
IMAGE_METADATA_FIELDS = ("model", "prompt_id", "group", "prompt", "seed", "steps", "guidance_scale", "scheduler",
                         "lora_scale", "nsfw_flagged")

def model_card(meta):
    """Half-page model card (C3.2): base model, data, training settings, results, limitations, license."""
    missing = [k for k in MODEL_CARD_FIELDS if k not in meta]
    if missing:
        raise ValueError(f"model card is missing {missing}")
    train = "\n".join(f"- {k}: {v}" for k, v in meta["training"].items())
    results = "\n".join(f"- {k}: {v}" for k, v in meta["results"].items())
    return f"""---
license: {meta["base_model_license"]}
base_model: {meta["base_model"]}
datasets:
- {meta["dataset_id"]}
tags: [stable-diffusion, lora, text-to-image, diffusers]
---
# {meta["repo_id"]} — run {meta["run"]}

**Base model:** {meta["base_model"]} · **Data:** {meta["dataset_id"]} (dataset license: {meta["dataset_license"]})

## Training settings
{train}

## Results
{results}

## Limitations
{meta["limitations"]}

## License and intended use
{meta["intended_use"]}
"""

def image_metadata(model, prompt_row, seed, nsfw_flagged):
    rec = {"model": model, "prompt_id": prompt_row["prompt_id"], "group": prompt_row["group"],
           "prompt": prompt_row["prompt"], "seed": int(seed), "steps": GEN["steps"],
           "guidance_scale": GEN["guidance_scale"], "scheduler": GEN["scheduler"],
           "lora_scale": GEN["lora_scale"] if model != "base" else None, "nsfw_flagged": nsfw_flagged}
    assert tuple(rec) == IMAGE_METADATA_FIELDS
    return rec

## 6. Safety analysis (C4)

Dataset license (copyrighted characters → non-commercial research only), NSFW filter on outputs (SD safety checker; flagged counts), memorization check (10 in-domain generations vs nearest training images by CLIP similarity; 3 closest pairs).

_Placeholder — not implemented yet._

## Appendix — Pre-Experiment Smoke Test (not an experiment; DECISIONS.md D-046)

Runs only when `RUN_SMOKE_TEST = True`. **P2-L\*** stages are CPU-safe (Mac). **P2-S\*** stages need CUDA and print `SKIPPED_LOCAL` otherwise — never counted as passed. No diffusion-model weights are downloaded by the local stages; temporary adapters live in the system temp directory and are deleted.

| Stage | Checks | Runs on |
|---|---|---|
| P2-L0 | versions, flags off, work dir outside the repo, config consistency | local + GPU |
| P2-L1 | synthetic dataset with planted corrupt / duplicate / near-duplicate / caption / size problems → exact removal counts | local + GPU |
| P2-L2 | real dataset card: availability, license, rows, schema; captions via column projection (no images downloaded) | local + GPU |
| P2-L3 | deterministic 90/10 split, disjoint, seed-sensitive, no near-duplicates across the split | local + GPU |
| P2-L4 | 512×512 preprocessing, tensor range, seeded data loader | local + GPU |
| P2-L5 | CLIP tokenizer (files only): 77 tokens, truncation, padding, real caption lengths | local + GPU |
| P2-L6 | A vs B differ only in rank/alpha; prompt table 10/5/5 from test captions | local + GPU |
| P2-L7 | metric plumbing without weights: KID inputs, CLIP-score definition, memorization NN, human-eval pairs + win rate | local + GPU |
| P2-L9 | exact LoRA sizes from the real SD 1.5 UNet config (meta device); tiny random UNet: 2 optimizer steps, save/reload adapter | local + GPU |
| P2-L8 | results/run-table/model-card/image-metadata schemas; no repo or professor-data changes; no `eval/`, no work dir | local + GPU (last) |
| P2-S0 | CUDA, GPU, packages, Hub reachability, HF token present | GPU |
| P2-S1 | real images → clean → 512 → VAE latents + text embeddings | GPU |
| P2-S2 | load SD 1.5, generate one base image | GPU |
| P2-S3 | LoRA r=4 / r=64 attached: total / trainable / % | GPU |
| P2-S4 | 3 optimizer steps per run and latent mode; finite loss, LoRA updated, frozen unchanged, peak VRAM, s/step, OOM fallback | GPU |
| P2-S5 | save adapter (temp) → reload → generate → delete | GPU |
| P2-S6 | CLIP score, LPIPS, KID on tiny inputs — **plumbing only** | GPU |
| P2-S7 | every model released; max peak ≤ 16 GB; no temp adapters left | GPU |

In [ ]:
# ===== Smoke test · harness utilities, P2-L0 environment/config, hygiene snapshot =====
# Status: PASS / FAIL / ERROR / INFO / SKIPPED_LOCAL (needs CUDA; pending the GPU machine — never counted as PASS).
import gc, json, shutil, statistics, sys, tempfile, time
import importlib.metadata as _md

SMOKE, LIFECYCLE = {}, []

def result(key, value):
    SMOKE[key] = value
    print(f"RESULT {key}: {value}")

def check(stage, name, ok, detail=""):
    SMOKE[f"{stage}.check.{name}"] = bool(ok)
    print(f"CHECK  {stage}.{name}: {'PASS' if ok else 'FAIL'}{(' — ' + str(detail)) if detail != '' else ''}")

def warn(msg):
    print(f"WARN   {msg}")

def gpu_state(tag):
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info(0)
    mb = lambda b: round(b / 2**20)
    return {"tag": tag, "allocated_mb": mb(torch.cuda.memory_allocated(0)), "reserved_mb": mb(torch.cuda.memory_reserved(0)),
            "peak_allocated_mb": mb(torch.cuda.max_memory_allocated(0)), "device_used_mb": mb(total - free)}

def snap(tag):
    s = gpu_state(tag); LIFECYCLE.append(s); return s

def _clear_gpu(reset_peak=False):
    gc.collect(); torch.cuda.empty_cache()
    if reset_peak:
        torch.cuda.reset_peak_memory_stats(0)

def run_stage(stage, fn, *args, needs_cuda=False):
    """Exceptions are recorded, never raised. CUDA stages: SKIPPED_LOCAL without a GPU; with one they are bracketed
    by baseline/released memory snapshots so released VRAM can be verified (sequential residency)."""
    if needs_cuda and not CUDA_AVAILABLE:
        result(f"{stage}.status", "SKIPPED_LOCAL")
        print(f"SKIPPED_LOCAL {stage}: requires a CUDA GPU — pending the GPU-machine run.")
        return None
    if needs_cuda:
        _clear_gpu(reset_peak=True); base = snap(f"{stage}.baseline")
    out, err, t0 = None, False, time.perf_counter()
    try:
        out = fn(stage, *args)
    except Exception as e:  # noqa: BLE001
        err = True
        result(f"{stage}.error", f"{type(e).__name__}: {e}"[:600])
    result(f"{stage}.wall_s", round(time.perf_counter() - t0, 2))
    if needs_cuda:
        _clear_gpu(); after = snap(f"{stage}.released")
        left = after["allocated_mb"] - base["allocated_mb"]
        check(stage, "vram_released", left <= SMOKE_RELEASE_TOLERANCE_MB, f"{left} MB left allocated")
    checks = [v for k, v in SMOKE.items() if k.startswith(f"{stage}.check.")]
    result(f"{stage}.status", "ERROR" if err else ("PASS" if checks and all(checks) else "FAIL" if checks else "INFO"))
    return out

def _version(dist):
    try:
        return _md.version(dist)
    except _md.PackageNotFoundError:
        return None

def _file_state(root, skip=(".git", ".venv", "__pycache__", ".ipynb_checkpoints")):
    root = Path(root)
    if not root.exists():
        return {}
    return {str(p.relative_to(root)): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(root.rglob("*")) if p.is_file() and not any(s in p.parts for s in skip)}

P2_LOCAL_PACKAGES = ["torch", "numpy", "pillow", "pandas", "datasets", "pyarrow", "huggingface-hub", "transformers",
                     "diffusers", "peft", "accelerate", "safetensors"]
P2_GPU_PACKAGES = ["torchmetrics", "torch-fidelity", "lpips"]   # metric weights are only loaded on the GPU machine

def l0_environment(stage):
    import platform
    result(f"{stage}.python", sys.version.split()[0])
    result(f"{stage}.platform", f"{platform.system()} {platform.machine()}")
    result(f"{stage}.local_packages", {d: _version(d) for d in P2_LOCAL_PACKAGES})
    result(f"{stage}.gpu_run_packages", {d: _version(d) for d in P2_GPU_PACKAGES})
    result(f"{stage}.cuda_available", CUDA_AVAILABLE)
    check(stage, "local_packages_installed", all(_version(d) for d in P2_LOCAL_PACKAGES),
          [d for d in P2_LOCAL_PACKAGES if not _version(d)] or "")
    check(stage, "real_training_and_evaluation_off", not RUN_TRAINING and not RUN_EVALUATION)
    check(stage, "work_dir_outside_repo", not P2_WORK_DIR.resolve().is_relative_to(REPO_DIR.resolve()), str(P2_WORK_DIR))
    check(stage, "effective_batch_divisible", TRAIN["effective_batch"] % TRAIN["micro_batch"] == 0)
    check(stage, "latent_mode_valid", TRAIN["latent_mode"] in LATENT_MODES, TRAIN["latent_mode"])
    check(stage, "hf_token_not_in_config", not any(isinstance(v, str) and v.startswith("hf_") for v in globals().values()))

# Watched for unexpected writes: the repository and (locally) the professor-provided folder next to it.
SMOKE_WATCH = {"repo": REPO_DIR} if ON_KAGGLE else {"repo": REPO_DIR, "professor_data": REPO_DIR.parent / "DATA266_Lab2_Part1"}

if RUN_SMOKE_TEST:
    SMOKE_FILES_BEFORE = {k: _file_state(v) for k, v in SMOKE_WATCH.items()}
    SMOKE_WORK_DIR_EXISTED = P2_WORK_DIR.exists()
    run_stage("P2-L0", l0_environment)
else:
    print("Smoke test skipped (RUN_SMOKE_TEST = False).")

In [ ]:
# ===== Smoke test · P2-L1 synthetic cleaning, P2-L2 dataset card + captions, P2-L3 split, P2-L4 preprocessing/loader, P2-L5 tokens =====
from PIL import ImageDraw

def _encode(img, fmt="PNG", **kw):
    buf = io.BytesIO(); img.save(buf, format=fmt, **kw); return buf.getvalue()

def synthetic_records():
    """24 in-memory rows: 18 distinct images (varied sizes and RGB/RGBA/L/P modes) + 6 planted problems.
    Expected removals: 2 decode_error, 1 exact_duplicate, 1 near_duplicate, 1 missing_caption, 1 too_small."""
    rng = np.random.default_rng(266)
    def scene(size):
        img = Image.new("RGB", size, tuple(int(c) for c in rng.integers(0, 256, 3)))
        d = ImageDraw.Draw(img)
        for _ in range(6):
            x0, y0 = int(rng.integers(0, size[0] // 2)), int(rng.integers(0, size[1] // 2))
            box = [x0, y0, x0 + int(rng.integers(size[0] // 4, size[0] // 2)), y0 + int(rng.integers(size[1] // 4, size[1] // 2))]
            (d.ellipse if rng.random() < 0.5 else d.rectangle)(box, fill=tuple(int(c) for c in rng.integers(0, 256, 3)))
        return img
    sizes = [(600, 500), (500, 800), (1024, 1024), (640, 360), (300, 700), (768, 512)]
    base = [scene(sizes[i % len(sizes)]) for i in range(18)]
    rgba = base[15].convert("RGBA"); rgba.putalpha(Image.new("L", rgba.size, 128)); base[15] = rgba
    base[16], base[17] = base[16].convert("L"), base[17].convert("P")
    recs = [{"idx": i, "image": {"bytes": _encode(im), "path": None}, "text": f"synthetic caption {i}"} for i, im in enumerate(base)]
    recs += [{"idx": 18, "image": {"bytes": _encode(base[0])[:200], "path": None}, "text": "truncated png"},
             {"idx": 19, "image": {"bytes": b"definitely not an image", "path": None}, "text": "not an image"},
             {"idx": 20, "image": {"bytes": _encode(base[0]), "path": None}, "text": "exact duplicate of 0"},
             {"idx": 21, "image": {"bytes": _encode(base[1], "JPEG", quality=85), "path": None}, "text": "near duplicate of 1"},
             {"idx": 22, "image": {"bytes": _encode(scene((600, 600))), "path": None}, "text": "   "},
             {"idx": 23, "image": {"bytes": _encode(scene((120, 100))), "path": None}, "text": "too small"}]
    return recs

def l1_synthetic_cleaning(stage):
    kept, removed, report = clean_records(synthetic_records())
    result(f"{stage}.report", report)
    result(f"{stage}.removed", removed)
    expected = {"total": 24, "kept": 18, "removed": 6, "missing_image": 0, "decode_error": 2, "too_small": 1,
                "missing_caption": 1, "exact_duplicate": 1, "near_duplicate": 1}
    check(stage, "removal_counts_as_planted", report == expected, report if report != expected else "")
    dup = {r["idx"]: r["duplicate_of"] for r in removed}
    check(stage, "first_occurrence_kept", dup.get(20) == 0 and dup.get(21) == 1, {20: dup.get(20), 21: dup.get(21)})
    check(stage, "all_kept_decode_to_rgb", all(decode_rgb(k["raw"])[0].mode == "RGB" for k in kept))
    pair_min = min((a["dhash"] ^ b["dhash"]).bit_count() for i, a in enumerate(kept) for b in kept[i + 1:])
    result(f"{stage}.min_dhash_distance_between_kept", pair_min)
    check(stage, "kept_images_not_near_duplicates", pair_min > NEAR_DUP_HAMMING_MAX)
    return kept

def l2_dataset_card(stage):
    """Card, license, schema and row counts from the Hub API + parquet footers; captions via column projection only
    (the image column is never downloaded). The rejected dataset's card is recorded for the audit trail (D-064)."""
    import pyarrow.parquet as pq
    from huggingface_hub import HfApi, HfFileSystem
    api, fs = HfApi(), HfFileSystem()
    def card(ds_id):
        info = api.dataset_info(ds_id)
        meta = info.card_data.to_dict() if info.card_data else {}
        lic = meta.get("license") or next((t.split(":", 1)[1] for t in (info.tags or []) if t.startswith("license:")), None)
        files = sorted(fs.glob(f"datasets/{ds_id}/**/*.parquet"))
        rows, cols, text_bytes = 0, None, 0
        for f in files:
            with fs.open(f, block_size=2**20) as fh:
                pf = pq.ParquetFile(fh)
                md, cols = pf.metadata, pf.schema_arrow.names
                rows += md.num_rows
                ci = next(i for i in range(md.num_columns) if md.schema.column(i).path == CAPTION_COLUMN)   # physical column
                text_bytes += sum(md.row_group(g).column(ci).total_compressed_size for g in range(md.num_row_groups))
        return {"id": info.id, "license_stated": lic, "rows": rows, "columns": cols, "parquet_files": len(files),
                "gated": info.gated, "disabled": info.disabled, "private": info.private,
                "caption_column_bytes": text_bytes}, files
    sel, files = card(DATASET_ID)
    result(f"{stage}.selected", sel)
    check(stage, "selected_available_with_image_and_text_columns",
          not sel["disabled"] and not sel["private"] and {IMAGE_COLUMN, CAPTION_COLUMN} <= set(sel["columns"] or []))
    check(stage, "selected_has_at_least_500_rows", sel["rows"] >= 500, sel["rows"])
    check(stage, "selected_license_stated_and_matches_config", sel["license_stated"] == DATASET_LICENSE, sel["license_stated"])
    result(f"{stage}.rejected_for_record", card(DATASET_REJECTED_ID)[0])
    captions = []
    for f in files:
        with fs.open(f, block_size=2**20) as fh:
            captions += pq.read_table(fh, columns=[CAPTION_COLUMN]).column(0).to_pylist()
    words = [len((c or "").split()) for c in captions]
    result(f"{stage}.selected_captions", {"n": len(captions), "empty": sum(not (c or "").strip() for c in captions),
                                          "unique": len(set(captions)), "words_mean": round(float(np.mean(words)), 2),
                                          "words_max": max(words), "examples": captions[:3]})
    check(stage, "selected_every_caption_present", all((c or "").strip() for c in captions))
    return {"rows": sel["rows"], "captions": captions}

def l3_split(stage, kept, n_rows):
    big = [{"idx": i} for i in range(n_rows)]             # dataset-sized id list (no images needed)
    tr, te = split_train_test(big)
    tr2, te2 = split_train_test(big)
    _, te_other = split_train_test(big, seed=SPLIT_SEED + 1)
    result(f"{stage}.sizes_for_dataset_rows", {"rows": n_rows, "train": len(tr), "test": len(te)})
    check(stage, "deterministic", (tr, te) == (tr2, te2))
    check(stage, "ninety_ten", len(te) == round(TEST_FRACTION * n_rows) and len(tr) + len(te) == n_rows, (len(tr), len(te)))
    check(stage, "disjoint_and_complete", not set(tr) & set(te) and set(tr) | set(te) == set(range(n_rows)))
    check(stage, "seed_matters", te != te_other)
    ktr, kte = split_train_test(kept)
    by = {k["idx"]: k for k in kept}
    leak = min((by[a]["dhash"] ^ by[b]["dhash"]).bit_count() for a in ktr for b in kte)
    result(f"{stage}.synthetic_split", {"train": ktr, "test": kte, "min_train_test_dhash_distance": leak})
    check(stage, "no_near_duplicate_across_split", leak > NEAR_DUP_HAMMING_MAX)
    return {"train_ids": tr, "test_ids": te}

def l4_preprocess_and_loader(stage, kept):
    from IPython.display import display
    imgs = [preprocess(decode_rgb(k["raw"])[0]) for k in kept]
    result(f"{stage}.input_sizes", sorted({k["size"] for k in kept}))
    check(stage, "all_512_rgb", all(im.size == (RESOLUTION, RESOLUTION) and im.mode == "RGB" for im in imgs))
    t = to_model_tensor(imgs[0])
    result(f"{stage}.tensor", {"shape": tuple(t.shape), "dtype": str(t.dtype), "min": round(float(t.min()), 3),
                               "max": round(float(t.max()), 3)})
    check(stage, "tensor_shape_and_range", tuple(t.shape) == (3, RESOLUTION, RESOLUTION) and t.min() >= -1 and t.max() <= 1)
    order = lambda seed: [b["idx"] for b in make_loader(kept, 4, seed)][:3]
    first = next(iter(make_loader(kept, 4, TRAIN["seed"])))
    result(f"{stage}.batch", {"pixel_values": tuple(first["pixel_values"].shape), "texts": len(first["text"])})
    check(stage, "batch_shape", tuple(first["pixel_values"].shape) == (4, 3, RESOLUTION, RESOLUTION))
    check(stage, "loader_order_seeded", order(TRAIN["seed"]) == order(TRAIN["seed"]) and order(TRAIN["seed"]) != order(TRAIN["seed"] + 1))
    grid = Image.new("RGB", (6 * 128, 3 * 128), "white")
    for i, im in enumerate(imgs[:18]):
        grid.paste(im.resize((128, 128)), ((i % 6) * 128, (i // 6) * 128))
    display(grid)

def l5_tokens(stage, captions):
    from transformers import CLIPTokenizer
    tok = CLIPTokenizer.from_pretrained(BASE_MODEL_ID, subfolder="tokenizer")   # tokenizer files only (no weights)
    result(f"{stage}.model_max_length", tok.model_max_length)
    check(stage, "max_length_77", tok.model_max_length == 77)
    ids = tokenize_captions(tok, captions)
    lengths = [len(x) for x in tok(list(captions), truncation=False).input_ids]
    result(f"{stage}.selected_caption_tokens", {"shape": tuple(ids.shape), "max_tokens": max(lengths),
                                                 "mean_tokens": round(float(np.mean(lengths)), 1),
                                                 "truncated": sum(n > tok.model_max_length for n in lengths)})
    check(stage, "ids_shape", tuple(ids.shape) == (len(captions), 77))
    long_ids = tokenize_captions(tok, ["word " * 200])[0]
    check(stage, "long_caption_truncated_with_eos", len(long_ids) == 77 and int(long_ids[-1]) == tok.eos_token_id)
    short = tokenize_captions(tok, ["a ninja"])[0]
    check(stage, "short_caption_padded", int(short[-1]) == tok.pad_token_id)

if RUN_SMOKE_TEST:
    L1_KEPT = run_stage("P2-L1", l1_synthetic_cleaning)
    L2_OUT = run_stage("P2-L2", l2_dataset_card)
    L3_SPLIT = run_stage("P2-L3", l3_split, L1_KEPT, L2_OUT["rows"]) if (L1_KEPT and L2_OUT) else None
    if L1_KEPT:
        run_stage("P2-L4", l4_preprocess_and_loader, L1_KEPT)
    run_stage("P2-L5", l5_tokens, (L2_OUT or {}).get("captions") or ["a red cartoon creature", "a blue bird with wings"])
else:
    print("Smoke test skipped (RUN_SMOKE_TEST = False).")

In [ ]:
# ===== Smoke test · P2-L6 experiment config + prompts, P2-L7 metric plumbing, P2-L9 tiny-UNet LoRA plumbing, P2-L8 schemas + hygiene =====
def l6_experiment_config(stage, split):
    a, b = run_config("A"), run_config("B")
    diff = sorted(k for k in set(a) | set(b) if a.get(k) != b.get(k))
    result(f"{stage}.run_A", a)
    result(f"{stage}.differences_A_vs_B", {k: (a[k], b[k]) for k in diff})
    check(stage, "only_rank_alpha_and_name_differ", diff == ["alpha", "rank", "run"], diff)
    check(stage, "alpha_equals_rank", all(r["alpha"] == r["rank"] for r in RUNS.values()))
    check(stage, "grad_accum_integer", a["grad_accum"] * a["micro_batch"] == a["effective_batch"])
    test_rows = [{"idx": i, "text": f"synthetic test caption {i}"} for i in split["test_ids"]]
    p1, p2 = draw_in_domain_prompts(test_rows), draw_in_domain_prompts(test_rows)
    check(stage, "in_domain_prompts_deterministic_and_from_test", p1 == p2 and {s for s, _ in p1} <= set(split["test_ids"]))
    table = build_prompt_table(p1, [f"PLACEHOLDER out-of-domain {i}" for i in range(5)],
                               [f"PLACEHOLDER compositional {i}" for i in range(5)])
    result(f"{stage}.prompt_table", {"rows": len(table), "groups": table["group"].value_counts().to_dict(),
                                     "columns": list(table.columns)})
    check(stage, "prompt_table_20_rows_10_5_5", len(table) == 20 and table["prompt_id"].is_unique)
    try:
        build_prompt_table(p1[:9], ["x"] * 5, ["y"] * 5); rejected = False
    except ValueError:
        rejected = True
    check(stage, "wrong_prompt_counts_rejected", rejected)
    check(stage, "one_generation_config_for_all_models", isinstance(GEN, dict) and GEN["seeds"] == (0, 1))
    return table

def l7_metric_plumbing(stage):
    check(stage, "kid_subset_size", kid_subset_size(83, 20) == 20 and kid_subset_size(83, 20, 50) == 20
          and kid_subset_size(10, 10, 4) == 4)
    try:
        kid_subset_size(1, 5); bad = False
    except ValueError:
        bad = True
    check(stage, "kid_rejects_too_few_images", bad)
    imgs = [Image.new("RGB", (RESOLUTION, RESOLUTION), c) for c in ("red", "green", "blue")]
    u8 = to_uint8_batch(imgs)
    check(stage, "kid_input_uint8_nchw", u8.dtype == torch.uint8 and tuple(u8.shape) == (3, 3, RESOLUTION, RESOLUTION))
    t = to_model_tensor(imgs[0])
    check(stage, "lpips_input_range", float(t.min()) >= -1 and float(t.max()) <= 1)
    e = torch.eye(4)
    s = clip_score_from_embeddings(torch.stack([e[0], e[0], e[0]]), torch.stack([e[0], e[1], -e[0]]))
    check(stage, "clip_score_definition", [round(v, 6) for v in s] == [100.0, 0.0, 0.0], s)
    g = torch.Generator().manual_seed(266)
    train = torch.nn.functional.normalize(torch.randn(50, 16, generator=g), dim=-1)
    gen = torch.nn.functional.normalize(train[[3, 7]] + 0.01 * torch.randn(2, 16, generator=g), dim=-1)
    sims, idx = nearest_training(gen, train)
    check(stage, "memorization_nearest_neighbour", idx == [3, 7] and min(sims) > 0.99, (idx, [round(x, 4) for x in sims]))
    cands = [(f"P{i:02d}", s) for i in range(1, 21) for s in GEN["seeds"]]
    pairs, pairs2 = draw_human_pairs(cands), draw_human_pairs(cands)
    sides = [p["left"] for p in pairs]
    result(f"{stage}.human_pairs", {"n": len(pairs), "final_on_left": sides.count("final"), "first": pairs[0]})
    check(stage, "human_pairs_25_unique_deterministic",
          pairs == pairs2 and len({(p["prompt_id"], p["seed"]) for p in pairs}) == HUMAN_EVAL_PAIRS)
    check(stage, "human_pairs_sides_randomised", 0 < sides.count("final") < HUMAN_EVAL_PAIRS)
    raters = ["r1", "r2", "r3"]
    ratings = [{"rater": r, "pair_id": p["pair_id"], "criterion": "domain_style",
                "choice": "left" if p["left"] == "final" else "right"} for r in raters for p in pairs]
    ratings += [{"rater": r, "pair_id": p["pair_id"], "criterion": "prompt_match", "choice": "left"} for r in raters for p in pairs]
    wr = human_win_rate(ratings, pairs)
    expected_left = sides.count("final") / len(pairs)
    result(f"{stage}.synthetic_win_rate", wr)
    check(stage, "win_rate_arithmetic", wr["domain_style"]["win_rate"] == 1.0
          and abs(wr["prompt_match"]["win_rate"] - expected_left) < 1e-12 and wr["domain_style"]["raters"] == 3)

def l9_tiny_unet_lora(stage):
    """CPU plumbing only: exact LoRA sizes from the real SD 1.5 UNet *config* (meta device, no weights), then
    2 optimizer steps + save/reload on a tiny randomly initialised UNet. Not training; says nothing about quality."""
    from accelerate import init_empty_weights
    from diffusers import DDPMScheduler, UNet2DConditionModel
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    cfg = UNet2DConditionModel.load_config(hf_hub_download(BASE_MODEL_ID, "unet/config.json"))
    sizes = {}
    for run, spec in RUNS.items():
        with init_empty_weights():
            meta_unet = UNet2DConditionModel.from_config(cfg)
            add_lora(meta_unet, spec["rank"], spec["alpha"])
        sizes[run] = count_params(meta_unet)
        del meta_unet
    result(f"{stage}.sd15_unet_lora_params_exact", sizes)
    check(stage, "rank64_has_16x_rank4_params", sizes["B"]["trainable"] == 16 * sizes["A"]["trainable"])
    torch.manual_seed(266)
    unet = UNet2DConditionModel(sample_size=8, in_channels=4, out_channels=4, layers_per_block=1,
                                block_out_channels=(32, 64), down_block_types=("CrossAttnDownBlock2D", "DownBlock2D"),
                                up_block_types=("UpBlock2D", "CrossAttnUpBlock2D"), cross_attention_dim=32,
                                attention_head_dim=8).requires_grad_(False)
    params = add_lora(unet, RUNS["A"]["rank"], RUNS["A"]["alpha"])
    frozen = [p for p in unet.parameters() if not p.requires_grad]
    frozen_before = [p.detach().clone() for p in frozen]
    lora_before = [p.detach().clone() for p in params]
    noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
    opt, sched = make_optimizer(params, total_steps=2)
    autocast, scaler = amp_tools("cpu")
    g = torch.Generator().manual_seed(266)
    mean, std = torch.randn(4, 4, 8, 8, generator=g), torch.rand(4, 4, 8, 8, generator=g) * 0.1
    z_a = latents_from_moments(mean, std, "cached_moments", g, 0.18215)
    z_b = latents_from_moments(mean, std, "cached_moments", g, 0.18215)
    check(stage, "cached_moments_resample_each_time", not torch.equal(z_a, z_b))
    check(stage, "cached_mean_deterministic", torch.equal(latents_from_moments(mean, std, "cached_mean", g, 0.18215),
                                                          latents_from_moments(mean, std, "cached_mean", g, 0.18215)))
    try:
        latents_from_moments(mean, std, "frozen_sample", g, 1.0); bad = False
    except ValueError:
        bad = True
    check(stage, "unknown_latent_mode_rejected", bad)
    emb = torch.randn(4, 77, 32, generator=g)
    losses = []
    for _ in range(2):   # 2 optimizer steps x 2 micro-batches of 2 (gradient accumulation path)
        micro = [(latents_from_moments(mean[i:i + 2], std[i:i + 2], "online", g, 0.18215), emb[i:i + 2]) for i in (0, 2)]
        losses.append(optimizer_step(unet, params, opt, sched, scaler, autocast, noise_scheduler, micro, g))
    result(f"{stage}.tiny_unet", {"params": count_params(unet), "loss_gradnorm_per_step": losses,
                                  "lr_after": sched.get_last_lr()})
    check(stage, "loss_and_grad_finite", all(np.isfinite(l) and np.isfinite(gn) and gn > 0 for l, gn in losses))
    check(stage, "only_lora_params_trainable_and_fp32", all(p.dtype == torch.float32 for p in params) and len(params) > 0)
    check(stage, "lora_params_updated", any(not torch.equal(a, p) for a, p in zip(lora_before, params)))
    check(stage, "frozen_params_unchanged", all(torch.equal(a, p) for a, p in zip(frozen_before, frozen)))
    tmp = Path(tempfile.mkdtemp(prefix="p2_smoke_lora_"))   # system temp dir, outside the repo
    try:
        save_lora(unet, tmp)
        saved = load_file(str(tmp / "pytorch_lora_weights.safetensors"))
        result(f"{stage}.saved_adapter", {"tensors": len(saved), "bytes": (tmp / "pytorch_lora_weights.safetensors").stat().st_size})
        check(stage, "adapter_saved_with_all_lora_tensors", len(saved) == len(params))
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
    check(stage, "temp_adapter_removed", not tmp.exists())
    return sizes

def l8_schemas_and_hygiene(stage):
    results = pd.DataFrame([{"model": m} for m in ("base", "A", "B")], columns=RESULTS_COLUMNS)
    runs = pd.DataFrame([{k: run_config(r).get(k) for k in RUN_TABLE_COLUMNS if k in run_config(r)} | {"run": r}
                         for r in RUNS], columns=RUN_TABLE_COLUMNS)
    result(f"{stage}.results_table_columns", list(results.columns))
    result(f"{stage}.run_table", runs.to_dict("records"))
    check(stage, "results_table_has_base_and_both_runs", list(results["model"]) == ["base", "A", "B"])
    check(stage, "run_table_has_rubric_fields", {"trainable_params", "learning_rate", "steps", "effective_batch",
                                                 "peak_gpu_mb", "train_time_s", "gpu_hours"} <= set(runs.columns))
    meta = {"repo_id": "PLACEHOLDER/p2-lora-r4", "run": "A", "base_model": BASE_MODEL_ID,
            "base_model_license": "creativeml-openrail-m", "dataset_id": DATASET_ID,
            "dataset_license": DATASET_LICENSE, "training": run_config("A"),
            "results": {"kid": "PENDING", "clip_score": "PENDING"}, "limitations": "PENDING",
            "intended_use": "Non-commercial research only (copyrighted characters)."}
    card = model_card(meta)
    check(stage, "model_card_has_yaml_and_sections", card.startswith("---\nlicense:") and all(
        s in card for s in ("base_model:", "## Training settings", "## Results", "## Limitations", "## License")))
    try:
        model_card({k: v for k, v in meta.items() if k != "limitations"}); rejected = False
    except ValueError:
        rejected = True
    check(stage, "model_card_rejects_missing_fields", rejected)
    rec = image_metadata("A", {"prompt_id": "P01", "group": "in_domain", "prompt": "x"}, 0, False)
    check(stage, "image_metadata_fields", tuple(rec) == IMAGE_METADATA_FIELDS)
    after = {k: _file_state(v) for k, v in SMOKE_WATCH.items()}
    for k in SMOKE_WATCH:
        b, a = SMOKE_FILES_BEFORE[k], after[k]
        changes = {"new": sorted(set(a) - set(b)), "deleted": sorted(set(b) - set(a)),
                   "modified": sorted(x for x in set(a) & set(b) if a[x] != b[x] and not x.endswith(".ipynb"))}
        result(f"{stage}.{k}_files", {"count": len(a), **changes})
        check(stage, f"{k}_unchanged", not any(changes.values()))
    check(stage, "eval_dir_not_created", not EVAL_DIR.exists())
    check(stage, "work_dir_not_created_by_smoke_test", P2_WORK_DIR.exists() == SMOKE_WORK_DIR_EXISTED)

if RUN_SMOKE_TEST:
    if L3_SPLIT:
        run_stage("P2-L6", l6_experiment_config, L3_SPLIT)
    run_stage("P2-L7", l7_metric_plumbing)
    L9_SIZES = run_stage("P2-L9", l9_tiny_unet_lora)
    run_stage("P2-L8", l8_schemas_and_hygiene)   # last local stage: hygiene covers everything above
else:
    print("Smoke test skipped (RUN_SMOKE_TEST = False).")

In [ ]:
# ===== Smoke test · GPU stages P2-S0 to P2-S3 (prepared; run only with CUDA — SKIPPED_LOCAL on the Mac) =====
def s0_gpu_environment(stage):
    import urllib.request
    props = torch.cuda.get_device_properties(0)
    result(f"{stage}.gpu", {"name": props.name, "total_mb": round(props.total_memory / 2**20), "count": torch.cuda.device_count(),
                            "capability": torch.cuda.get_device_capability(0), "torch_cuda": torch.version.cuda})
    result(f"{stage}.packages", {d: _version(d) for d in P2_LOCAL_PACKAGES + P2_GPU_PACKAGES})
    check(stage, "gpu_run_packages_installed", all(_version(d) for d in P2_GPU_PACKAGES))
    try:
        urllib.request.urlopen("https://huggingface.co", timeout=10); hub = True
    except Exception:  # noqa: BLE001
        hub = False
    check(stage, "huggingface_reachable", hub)
    if not os.environ.get(HF_TOKEN_ENV):
        warn(f"${HF_TOKEN_ENV} is not set: downloads work, uploads (models/images) will not.")
    if torch.cuda.device_count() > 1:
        warn(f"{torch.cuda.device_count()} GPUs visible; everything is pinned to {DEVICE} (one-GPU rule).")
    if props.total_memory / 2**20 > VRAM_LIMIT_MB:
        warn(f"GPU has more than 16 GB: every peak must still be <= {VRAM_LIMIT_MB} MB (checked per stage).")

def s1_real_batch(stage):
    """Real images: clean -> 512 -> VAE moments / latents and cached text embeddings for a tiny batch."""
    from IPython.display import display
    t0 = time.perf_counter()
    kept, removed, report = clean_records(load_records(limit=4 * SMOKE_GPU_IMAGES))
    result(f"{stage}.load_and_clean_s", round(time.perf_counter() - t0, 1))
    result(f"{stage}.clean_report_first_rows", report)
    rows = kept[:SMOKE_GPU_IMAGES]
    check(stage, "enough_clean_images", len(rows) == SMOKE_GPU_IMAGES, len(rows))
    batch = collate_batch([CaptionedImages(rows)[i] for i in range(len(rows))])
    comps = load_sd_components(need=("tokenizer", "text_encoder", "vae"))
    emb = encode_text(comps["tokenizer"], comps["text_encoder"], batch["text"])
    torch.cuda.synchronize(); t0 = time.perf_counter()
    mean, std = vae_moments(comps["vae"], batch["pixel_values"][:TRAIN["micro_batch"]])
    torch.cuda.synchronize(); vae_s = time.perf_counter() - t0
    mean, std = vae_moments(comps["vae"], batch["pixel_values"])
    g = torch.Generator(device=DEVICE).manual_seed(TRAIN["seed"])
    z = latents_from_moments(mean, std, "online", g, comps["vae"].config.scaling_factor)
    result(f"{stage}.shapes", {"text_emb": tuple(emb.shape), "latents": tuple(z.shape)})
    result(f"{stage}.vae_encode_s_per_micro_batch", round(vae_s, 4))
    check(stage, "text_emb_shape", tuple(emb.shape) == (len(rows), 77, 768))
    check(stage, "latent_shape", tuple(z.shape) == (len(rows), 4, RESOLUTION // 8, RESOLUTION // 8))
    check(stage, "finite", bool(torch.isfinite(emb).all() and torch.isfinite(z).all()))
    result(f"{stage}.peak", snap(f"{stage}.peak"))
    grid = Image.new("RGB", (4 * 128, 2 * 128), "white")
    for i, r in enumerate(rows[:8]):
        grid.paste(preprocess(decode_rgb(r["raw"])[0]).resize((128, 128)), ((i % 4) * 128, (i // 4) * 128))
    display(grid)
    del comps, emb, mean, std, z
    return {"rows": rows, "pixel_values": batch["pixel_values"], "text": batch["text"]}

def s2_base_generation(stage, s1):
    from IPython.display import display
    t0 = time.perf_counter(); pipe = build_pipeline(); result(f"{stage}.load_s", round(time.perf_counter() - t0, 1))
    snap(f"{stage}.loaded")
    torch.cuda.synchronize(); t0 = time.perf_counter()
    img, nsfw = generate(pipe, s1["text"][0], seed=GEN["seeds"][0])
    torch.cuda.synchronize(); secs = time.perf_counter() - t0
    peak = snap(f"{stage}.peak")
    result(f"{stage}.generation", {"prompt": s1["text"][0], "seconds": round(secs, 2), "nsfw_flagged": nsfw,
                                   "size": img.size, "steps": GEN["steps"]})
    result(f"{stage}.peak", peak)
    check(stage, "image_512", img.size == (RESOLUTION, RESOLUTION))
    check(stage, "image_not_blank", float(np.asarray(img).std()) > 1.0)
    check(stage, "peak_within_16gb", peak["peak_allocated_mb"] <= VRAM_LIMIT_MB)
    display(img)
    del pipe
    return img

def s3_adapter_sizes(stage):
    out = {}
    for run, spec in RUNS.items():
        comps = load_sd_components(need=("unet",))
        add_lora(comps["unet"], spec["rank"], spec["alpha"])
        out[run] = count_params(comps["unet"])
        del comps; _clear_gpu()
    result(f"{stage}.params", out)
    local = SMOKE.get("P2-L9.sd15_unet_lora_params_exact")
    if local:
        check(stage, "matches_local_exact_counts", all(out[r]["trainable"] == local[r]["trainable"] for r in RUNS))
    check(stage, "rank64_is_16x_rank4", out["B"]["trainable"] == 16 * out["A"]["trainable"])
    return out

if RUN_SMOKE_TEST:
    run_stage("P2-S0", s0_gpu_environment, needs_cuda=True)
    S1_GPU = run_stage("P2-S1", s1_real_batch, needs_cuda=True)
    S2_IMG = run_stage("P2-S2", s2_base_generation, S1_GPU, needs_cuda=True) if (S1_GPU or not CUDA_AVAILABLE) else None
    run_stage("P2-S3", s3_adapter_sizes, needs_cuda=True)
else:
    print("Smoke test skipped (RUN_SMOKE_TEST = False).")

In [ ]:
# ===== Smoke test · GPU stages P2-S4 to P2-S7 (prepared; run only with CUDA) + status summary =====
def s4_train_steps(stage, s1):
    """SMOKE_TRAIN_STEPS optimizer steps per run and latent mode at the configured effective batch.
    Micro-batch falls back 4 -> 2 -> 1 on OOM (effective batch unchanged). Measures finite loss, gradient flow,
    frozen weights unchanged, peak VRAM and s/step for 'online' vs 'cached_moments' latents (decides D-055)."""
    comps = load_sd_components(need=("tokenizer", "text_encoder"))
    emb_all = encode_text(comps["tokenizer"], comps["text_encoder"], s1["text"]).detach()
    del comps; _clear_gpu()                                    # text encoder freed: embeddings cached (D-055)
    vae = load_sd_components(need=("vae",))["vae"]
    px, eff, sf = s1["pixel_values"], TRAIN["effective_batch"], vae.config.scaling_factor
    lora_dir, summary = None, {}
    for run, spec in RUNS.items():
        for micro in [m for m in SMOKE_MICRO_BATCHES if eff % m == 0]:
            unet = ns = params = opt = sched = cached = None
            torch.cuda.reset_peak_memory_stats(0)
            try:
                comps = load_sd_components(need=("unet", "noise_scheduler"))
                unet, ns = comps["unet"], comps["noise_scheduler"]; del comps
                if TRAIN["gradient_checkpointing"]:
                    unet.enable_gradient_checkpointing()
                params = add_lora(unet, spec["rank"], spec["alpha"])
                probe = next(p for p in unet.parameters() if not p.requires_grad and p.ndim > 1)
                probe_before, lora_before = probe.detach().clone(), [p.detach().clone() for p in params]
                opt, sched = make_optimizer(params, total_steps=2 * SMOKE_TRAIN_STEPS)
                autocast, scaler = amp_tools(DEVICE)
                g = torch.Generator(device=DEVICE).manual_seed(TRAIN["seed"])
                cached = vae_moments(vae, px)
                timing = {}
                for mode in ("online", "cached_moments"):
                    times, logs = [], []
                    for step in range(SMOKE_TRAIN_STEPS):
                        sel = [(step * eff + j) % len(px) for j in range(eff)]
                        torch.cuda.synchronize(); t0 = time.perf_counter()
                        mbs = []
                        for k in range(0, eff, micro):
                            ids = sel[k:k + micro]
                            mean, std = vae_moments(vae, px[ids]) if mode == "online" else (cached[0][ids], cached[1][ids])
                            mbs.append((latents_from_moments(mean, std, mode, g, sf), emb_all[ids]))
                        logs.append(optimizer_step(unet, params, opt, sched, scaler, autocast, ns, mbs, g))
                        torch.cuda.synchronize(); times.append(time.perf_counter() - t0)
                    timing[mode] = {"s_per_step": round(statistics.median(times[1:] or times), 3), "loss_gradnorm": logs}
                peak = snap(f"{stage}.{run}.micro{micro}.peak")
                s_step = timing[TRAIN["latent_mode"]]["s_per_step"]
                summary[run] = {"micro_batch": micro, "grad_accum": eff // micro, "peak_allocated_mb": peak["peak_allocated_mb"],
                                "timing": timing, "configured_latent_mode": TRAIN["latent_mode"],
                                "projected_train_hours_ESTIMATE": round(s_step * TRAIN["max_train_steps"] / 3600, 2),
                                "cached_vs_online_speedup": round(timing["online"]["s_per_step"]
                                                                  / max(timing["cached_moments"]["s_per_step"], 1e-9), 2)}
                result(f"{stage}.{run}", summary[run])
                all_logs = timing["online"]["loss_gradnorm"] + timing["cached_moments"]["loss_gradnorm"]
                check(stage, f"{run}_loss_finite", all(np.isfinite(l) for l, _ in all_logs))
                check(stage, f"{run}_lora_updated", any(not torch.equal(a, p) for a, p in zip(lora_before, params)))
                check(stage, f"{run}_frozen_weights_unchanged", torch.equal(probe_before, probe))
                check(stage, f"{run}_peak_within_16gb", peak["peak_allocated_mb"] <= VRAM_LIMIT_MB)
                if run == "A":
                    lora_dir = Path(tempfile.mkdtemp(prefix="p2_smoke_lora_")); save_lora(unet, lora_dir)
                break
            except torch.cuda.OutOfMemoryError:
                result(f"{stage}.{run}.micro{micro}", "OOM — trying a smaller micro-batch")
            finally:
                unet = ns = params = opt = sched = cached = None
                _clear_gpu()
        check(stage, f"{run}_fits_some_micro_batch", run in summary)
    del vae, emb_all
    return lora_dir

def s5_adapter_reload(stage, lora_dir, s1, base_img):
    from IPython.display import display
    try:
        files = {f.name: f.stat().st_size for f in Path(lora_dir).glob("*.safetensors")}
        result(f"{stage}.saved_files", files)
        check(stage, "adapter_file_saved", bool(files))
        pipe = build_pipeline(lora_dir=lora_dir)
        img, nsfw = generate(pipe, s1["text"][0], GEN["seeds"][0], lora=True)
        result(f"{stage}.generation", {"nsfw_flagged": nsfw, "size": img.size,
                                       "mean_abs_diff_vs_base": round(float(np.abs(np.asarray(img, dtype=np.float32)
                                                                     - np.asarray(base_img, dtype=np.float32)).mean()), 3)
                                       if base_img is not None else None})
        check(stage, "reloaded_adapter_generates_512", img.size == (RESOLUTION, RESOLUTION))
        if base_img is not None:
            pair = Image.new("RGB", (2 * RESOLUTION, RESOLUTION)); pair.paste(base_img, (0, 0)); pair.paste(img, (RESOLUTION, 0))
            display(pair.resize((RESOLUTION, RESOLUTION // 2)))
        del pipe
        return img
    finally:
        shutil.rmtree(lora_dir, ignore_errors=True)
        check(stage, "temp_adapter_removed", not Path(lora_dir).exists())

def s6_metric_plumbing(stage, s1, base_img, lora_img):
    """Plumbing only: tiny inputs; none of these numbers is a metric result."""
    real = [preprocess(decode_rgb(r["raw"])[0]) for r in s1["rows"]]
    gens = [im for im in (base_img, lora_img) if im is not None] or real[:2]
    clip = ClipModel()
    scores = clip.scores(gens, [s1["text"][0]] * len(gens))
    sims, idx = nearest_training(clip.image_embeddings(gens), clip.image_embeddings(real))
    result(f"{stage}.clip_score_PLUMBING_ONLY", [round(s, 3) for s in scores])
    result(f"{stage}.nearest_training_PLUMBING_ONLY", {"cosine": [round(s, 4) for s in sims], "index": idx})
    check(stage, "clip_scores_in_range", all(0 <= s <= 100 for s in scores))
    del clip; _clear_gpu()
    lp = lpips_model()
    d, d_same = lpips_distance(lp, real[0], real[1]), lpips_distance(lp, real[0], real[0])
    result(f"{stage}.lpips_PLUMBING_ONLY", {"different": round(d, 4), "identical": round(d_same, 6)})
    check(stage, "lpips_identical_zero_different_positive", d_same < 1e-4 and d > 0)
    del lp; _clear_gpu()
    k_mean, k_std = kid(real[:SMOKE_KID_N], real[SMOKE_KID_N:2 * SMOKE_KID_N], subset_size=SMOKE_KID_N)
    result(f"{stage}.kid_PLUMBING_ONLY_not_a_metric", {"mean": k_mean, "std": k_std, "n": SMOKE_KID_N})
    check(stage, "kid_finite", np.isfinite(k_mean) and np.isfinite(k_std))

def s7_release(stage):
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(LIFECYCLE).set_index("tag"))
    released = {k.split(".")[0]: v for k, v in SMOKE.items() if k.endswith(".check.vram_released")}
    result(f"{stage}.released_per_stage", released)
    result(f"{stage}.max_peak_allocated_mb", max(s["peak_allocated_mb"] for s in LIFECYCLE))
    check(stage, "every_gpu_stage_released_memory", bool(released) and all(released.values()))
    check(stage, "max_peak_within_16gb", max(s["peak_allocated_mb"] for s in LIFECYCLE) <= VRAM_LIMIT_MB)
    check(stage, "no_temp_adapters_left", not list(Path(tempfile.gettempdir()).glob("p2_smoke_lora_*")))

if RUN_SMOKE_TEST:
    S4_LORA_DIR = run_stage("P2-S4", s4_train_steps, S1_GPU, needs_cuda=True) if (S1_GPU or not CUDA_AVAILABLE) else None
    S5_IMG = run_stage("P2-S5", s5_adapter_reload, S4_LORA_DIR, S1_GPU, S2_IMG, needs_cuda=True) \
        if (S4_LORA_DIR or not CUDA_AVAILABLE) else None
    run_stage("P2-S6", s6_metric_plumbing, S1_GPU, S2_IMG, S5_IMG, needs_cuda=True) if (S1_GPU or not CUDA_AVAILABLE) else None
    run_stage("P2-S7", s7_release, needs_cuda=True)
    print("\n===== P2 SMOKE TEST STAGE STATUS (machine-generated) =====")
    print(json.dumps({k.rsplit(".", 1)[0]: v for k, v in SMOKE.items() if k.endswith(".status")}, indent=1))
else:
    print("Smoke test skipped (RUN_SMOKE_TEST = False).")